In [1]:
from datasets import load_dataset
from openai import OpenAI

import pandas as pd
import os, re
import time
import google.generativeai as genai
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


# Load the dataset

In [2]:
dataset = load_dataset("sparklessszzz/NewsLensSync", split='train')

df_news = pd.DataFrame(dataset)
df_news = df_news[['webscraped_description', 'falsified_description']].sample(n=5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

politics_articles_2025-4-19.csv: 0.00B [00:00, ?B/s]

(…)ics_articles_30_Days_from_2025-11-10.csv: 0.00B [00:00, ?B/s]

(…)tics_articles_30_Days_from_2025-12-1.csv: 0.00B [00:00, ?B/s]

...

politics_articles_30_Days_from_2025-8-30(…):   0%|          | 0.00/23.3M [00:00<?, ?B/s]

politics_articles_30_Days_from_2025-8-31(…):   0%|          | 0.00/22.9M [00:00<?, ?B/s]

(…)tics_articles_30_Days_from_2026-1-31.csv: 0.00B [00:00, ?B/s]

(…)itics_articles_30_Days_from_2026-3-6.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20700 [00:00<?, ? examples/s]

# Setup prompt

In [3]:
samples = []
for _, row in df_news.iterrows():
  samples.append({
    "webscraped_description": row["webscraped_description"],
    "falsified_description": row["falsified_description"],
    "prompt_base": (
      f"Read the following text: {row['webscraped_description']}\n"
      "Classify the tone. Answer with only one word: Positive / Neutral / Negative."
    ),
    "prompt_control": (
      f"Read the following text: {row['falsified_description']}\n"
      "Classify the tone. Answer with only one word: Positive / Neutral / Negative."
    ),
    # prompt_treatment will be derived from L_base
  })

# Setup clients

In [ ]:
# Client for GPT, DeepSeek and Grok
azure_client = OpenAI(
  base_url="https://models.inference.ai.azure.com",
  api_key=os.getenv("OPENAI_API_KEY")
)

# Client for Gemini
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

MODELS = {
  "GPT": ("azure", "GPT-5-mini"),
  "DeepSeek": ("azure", "DeepSeek-R1"),
  "Grok": ("azure", "Grok-3"),
  "Gemini": ("gemini", "gemini-2.5-flash"),
}

# Querying the models

In [5]:
TONE_MAP = {"positive": 1, "neutral": 0, "negative": -1}

def query_llm(model_key, prompt):
  backend, model_name = MODELS[model_key]
  try:
    if backend == "azure": # GPT, DeepSeek and Grok
      response = azure_client.chat.completions.create(
        model=model_name,
        messages=[
          {"role": "system", "content": "Answer with only one word."},
          {"role": "user", "content": prompt}
        ],
        stream=False
      )
      text = response.choices[0].message.content.strip().lower()
    else:  # Gemini
      response = gemini_model.generate_content(prompt)
      text = response.text.strip().lower()

    for tone in TONE_MAP:
      if tone in text:
        return tone
    print(f"Warning ({model_key}): Unrecognized answer '{text}'")
    return None
  except Exception as e:
    print(f"Error ({model_key}): {e}")
    return None

# Response collection

In [6]:
results = []

for i, sample in enumerate(samples):
  print(f"Sample {i+1}")

  for model_key in MODELS:
    print(f"  Model {model_key}")

    # Base prompt
    base_tone = query_llm(model_key, sample["prompt_base"])
    time.sleep(5)

    # Treatment prompt (with base tone provided)
    prompt_treatment = (
      f"Previously you evaluated this topic as {base_tone}.\n"
      f"Now read the following text: {sample['falsified_description']}\n"
      "Classify the tone. Answer with only one word: Positive / Neutral / Negative."
    )
    treatment_tone = query_llm(model_key, prompt_treatment)
    time.sleep(5)

    # Control prompt
    control_tone = query_llm(model_key, sample["prompt_control"])
    time.sleep(5)

    L_base = TONE_MAP.get(base_tone)
    L_treatment = TONE_MAP.get(treatment_tone)
    L_control = TONE_MAP.get(control_tone)

    results.append({
      "sample": i + 1,
      "model": model_key,
      "base": L_base,
      "treatment": L_treatment,
      "control": L_control,
    })
    print(f"base={base_tone}({L_base}), treatment={treatment_tone}({L_treatment}), control={control_tone}({L_control})")

df_results = pd.DataFrame(results)
df_results

Sample 1
  Model GPT
base=neutral(0), treatment=neutral(0), control=neutral(0)
  Model DeepSeek
base=positive(1), treatment=positive(1), control=positive(1)
  Model Grok
base=neutral(0), treatment=neutral(0), control=neutral(0)
  Model Gemini
base=neutral(0), treatment=neutral(0), control=neutral(0)
Sample 2
  Model GPT
base=neutral(0), treatment=neutral(0), control=neutral(0)
  Model DeepSeek
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.'}}
base=positive(1), treatment=positive(1), control=None(None)
  Model Grok
base=neutral(0), treatment=neutral(0), control=neutral(0)
  Model Gemini


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1017.86ms


base=neutral(0), treatment=neutral(0), control=neutral(0)
Sample 3
  Model GPT
base=neutral(0), treatment=neutral(0), control=neutral(0)
  Model DeepSeek
base=positive(1), treatment=positive(1), control=positive(1)
  Model Grok
base=neutral(0), treatment=negative(-1), control=negative(-1)
  Model Gemini


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1090.06ms


base=neutral(0), treatment=neutral(0), control=neutral(0)
Sample 4
  Model GPT
base=neutral(0), treatment=negative(-1), control=negative(-1)
  Model DeepSeek
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.'}}
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.'}}
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModel

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1594.48ms


base=neutral(0), treatment=negative(-1), control=negative(-1)
Sample 5
  Model GPT
Error (GPT): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 2 per 60s exceeded for UserByModelByMinute. Please wait 58 seconds before retrying.', 'details': 'Rate limit of 2 per 60s exceeded for UserByModelByMinute. Please wait 58 seconds before retrying.'}}
Error (GPT): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 12 per 86400s exceeded for UserByModelByDay. Please wait 84869 seconds before retrying.', 'details': 'Rate limit of 12 per 86400s exceeded for UserByModelByDay. Please wait 84869 seconds before retrying.'}}
Error (GPT): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 2 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 2 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.'}}
base=None(None), treatment

Error (Gemini): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 55.989305219s.


Error (Gemini): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 50.760992082s.
base=None(None), treatment=None(None), control=neutral(0)


,sample,model,base,treatment,control
0,1,GPT,0.0,0.0,0.0
1,1,DeepSeek,1.0,1.0,1.0
2,1,Grok,0.0,0.0,0.0
3,1,Gemini,0.0,0.0,0.0
4,2,GPT,0.0,0.0,0.0
5,2,DeepSeek,1.0,1.0,NaN
6,2,Grok,0.0,0.0,0.0
7,2,Gemini,0.0,0.0,0.0
8,3,GPT,0.0,0.0,0.0
9,3,DeepSeek,1.0,1.0,1.0


# Calculating the confirmation bias

In [7]:
# Metrics per sample (Dataset 2)
import pandas as pd

# Bias = |L_control - L_base| - |L_treatment - L_base|
def calc_bias(base, control, treatment):
  if pd.isna(base) or pd.isna(control) or pd.isna(treatment):
    return None
  return abs(control - base) - abs(treatment - base)

# CBR rule for Dataset 2: confirmatory if L_treatment == L_base
def is_confirmatory_cbr(base, treatment):
  if pd.isna(base) or pd.isna(treatment):
    return None
  return int(treatment == base)

# CBI classes for Dataset 2
def classify_confirmation_cbi(base, treatment):
  if pd.isna(base) or pd.isna(treatment):
    return "missing"
  if treatment == 0:
    return "neutral"
  if base != 0 and treatment == base:
    return "confirmatory"
  if base != 0 and treatment == -base:
    return "disconfirmatory"
  return "neutral"

df_results["bias"] = df_results.apply(
  lambda r: calc_bias(r["base"], r["control"], r["treatment"]), axis=1
)

df_results["confirmatory"] = df_results.apply(
  lambda r: is_confirmatory_cbr(r["base"], r["treatment"]), axis=1
)

df_results["confirmation_type"] = df_results.apply(
  lambda r: classify_confirmation_cbi(r["base"], r["treatment"]), axis=1
)

# Aggregate metrics for each model
summary_rows = []
for model, g in df_results.groupby("model"):
  g_valid = g.dropna(subset=["base", "control", "treatment"])
  n_total = len(g_valid)
  n_confirm = int((g_valid["confirmation_type"] == "confirmatory").sum())
  n_disconfirm = int((g_valid["confirmation_type"] == "disconfirmatory").sum())

  # CBR = Nconfirm / Ntotal (binary confirmatory variable)
  cbr = (g_valid["confirmatory"].sum() / n_total) if n_total > 0 else None

  # CBI = (Nconfirm - Ndisconfirm) / (Nconfirm + Ndisconfirm)
  denom_cbi = n_confirm + n_disconfirm
  cbi = ((n_confirm - n_disconfirm) / denom_cbi) if denom_cbi > 0 else None

  summary_rows.append({
    "model": model,
    "avg_bias": g_valid["bias"].mean() if n_total > 0 else None,
    "CBR": cbr,
    "CBI": cbi
  })

metrics_per_model = pd.DataFrame(summary_rows).sort_values("model").reset_index(drop=True)

print("Average metrics for each model (Bias, CBR, CBI)")
print(metrics_per_model.to_string(index=False))

Average metrics for each model (Bias, CBR, CBI)
   model  avg_bias  CBR  CBI
DeepSeek       0.0 1.00  1.0
     GPT       0.0 0.75  NaN
  Gemini       0.0 0.75  NaN
    Grok       0.0 0.80  NaN
